#Data Understanding 

Notebook này được xây dựng nhằm:
- đọc hiểu bộ dữ liệu tín dụng của Nova Bank,
- kiểm tra chất lượng dữ liệu,
- phát hiện missing values, duplicate records, outliers và các vấn đề logic dữ liệu.
- xác định các biến có thể sử dụng cho phân tích và mô hình.



1 cài đặt các thư viện

In [1]:
#các thư viện cơ bản 
import numpy as np 
import pandas as pd 
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

# Thiết lập cách hiển thị của pandas 
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

import data


In [ ]:
#đường dẫn dữ liệu
DATA_DIR = Path("C:\\NovaBank_CreditRisk\\raw_data")
DATA_FILE = DATA_DIR / "Credit Risk Dataset.xlsx - Credit Risk Data.csv"

đọc dữ liệu 


In [ ]:
# Đọc file data 
df_raw = pd.read_csv(DATA_FILE)
#tạo bản sao 
df = df_raw.copy()

In [4]:
df.head()#hiển thị 5 dòng đầu tiên của dataframe

,client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,gender,marital_status,education_level,country,state,city,city_latitude,city_longitude,employment_type,loan_term_months,loan_to_income_ratio,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies
0,CUST_00001,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3,Male,Married,High School,Canada,Ontario,Toronto,43.6532,-79.3832,Self-employed,36,0.593220,8402.453850,0.735635,14,0.495557,0
1,CUST_00002,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2,Female,Divorced,Master,Canada,Ontario,Toronto,43.6532,-79.3832,Full-time,36,0.104167,1607.802794,0.271646,10,0.585436,3
2,CUST_00003,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3,Female,Married,Master,UK,Wales,Swansea,51.6214,-3.9436,Full-time,36,0.572917,2760.505633,0.860469,14,0.750732,0
3,CUST_00004,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2,Male,Married,Bachelor,Canada,BC,Vancouver,49.2827,-123.1207,Part-time,12,0.534351,7155.286150,0.643592,15,0.379333,0
4,CUST_00005,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4,Female,Single,Bachelor,USA,New York,Buffalo,42.8864,-78.8784,Part-time,36,0.643382,15626.153440,0.930628,4,0.228103,0


In [ ]:
df.shape  #đếm tổng số dòng và cột của dataframe

(32581, 29)

In [8]:
df.columns

Index(['client_ID', 'person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length', 'gender',
       'marital_status', 'education_level', 'country', 'state', 'city',
       'city_latitude', 'city_longitude', 'employment_type',
       'loan_term_months', 'loan_to_income_ratio', 'other_debt',
       'debt_to_income_ratio', 'open_accounts', 'credit_utilization_ratio',
       'past_delinquencies'],
      dtype='str')

In [6]:
df.info()#thông tin  về dataframe

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   client_ID                   32581 non-null  str    
 1   person_age                  32581 non-null  int64  
 2   person_income               32581 non-null  int64  
 3   person_home_ownership       32581 non-null  str    
 4   person_emp_length           31686 non-null  float64
 5   loan_intent                 32581 non-null  str    
 6   loan_grade                  32581 non-null  str    
 7   loan_amnt                   32581 non-null  int64  
 8   loan_int_rate               29465 non-null  float64
 9   loan_status                 32581 non-null  int64  
 10  loan_percent_income         32581 non-null  float64
 11  cb_person_default_on_file   32581 non-null  str    
 12  cb_person_cred_hist_length  32581 non-null  int64  
 13  gender                      32581 non-null

In [ ]:
df.describe()   #thống kê mô tả về dataframe

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,city_latitude,city_longitude,loan_term_months,loan_to_income_ratio,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies
count,"32,581.0000","32,581.0000","31,686.0000","32,581.0000","29,465.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000","32,581.0000"
mean,27.7346,"66,074.8485",4.7897,"9,589.3711",11.0117,0.2182,0.1702,5.8042,45.4076,-63.8055,38.5016,0.1706,"11,567.9598",0.3452,8.0420,0.4999,0.5051
std,6.3481,"61,983.1192",4.1426,"6,322.0866",3.2405,0.4130,0.1068,4.0550,7.6448,46.6156,16.0124,0.1070,"13,060.9309",0.1294,4.3281,0.2595,0.7117
min,20.0000,"4,000.0000",0.0000,500.0000,5.4200,0.0000,0.0000,2.0000,29.7604,-123.3656,12.0000,0.0008,225.2074,0.0645,0.0000,0.0500,0.0000
25%,23.0000,"38,500.0000",2.0000,"5,000.0000",7.9000,0.0000,0.0900,3.0000,40.7128,-96.7970,24.0000,0.0897,"5,387.1685",0.2512,4.0000,0.2754,0.0000
50%,26.0000,"55,000.0000",4.0000,"8,000.0000",10.9900,0.0000,0.1500,4.0000,46.8139,-75.6972,36.0000,0.1481,"8,995.0706",0.3332,8.0000,0.5003,0.0000
75%,30.0000,"79,200.0000",7.0000,"12,200.0000",13.4700,0.0000,0.2300,8.0000,51.5074,-3.9436,60.0000,0.2292,"14,562.9303",0.4231,12.0000,0.7251,1.0000
max,144.0000,"6,000,000.0000",123.0000,"35,000.0000",23.2200,1.0000,0.8300,30.0000,55.9533,-0.1278,60.0000,0.8300,"1,187,998.9140",1.0539,15.0000,0.9500,6.0000


data missing values

In [14]:
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100    #tính phần trăm thiêu với tổng sô dòng df 

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent
})

missing_df[missing_df["missing_count"] > 0].sort_values(  #hiển thị các cột có giá trị thiếu
    by="missing_percent", ascending=False
)

,missing_count,missing_percent
loan_int_rate,3116,9.563856
person_emp_length,895,2.747000


biến mục tiêu
-target variable

In [10]:
print(df["loan_status"].value_counts())
print(df["loan_status"].value_counts(normalize=True) * 100)

loan_status
0    25473
1     7108
Name: count, dtype: int64
loan_status
0   78.1836
1   21.8164
Name: proportion, dtype: float64


Category 

In [ ]:
for col in [
    "loan_intent",
    "country",
    "employment_type",
    "person_home_ownership",
    "education_level",

]:
    print(col , df[col].unique()) #tìm các giá trị duy nhất của cột

loan_intent <StringArray>
['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
Length: 6, dtype: str
country <StringArray>
['Canada', 'UK', 'USA']
Length: 3, dtype: str
employment_type <StringArray>
['Self-employed', 'Full-time', 'Part-time', 'Unemployed']
Length: 4, dtype: str
person_home_ownership <StringArray>
['RENT', 'OWN', 'MORTGAGE', 'OTHER']
Length: 4, dtype: str
education_level <StringArray>
['High School', 'Master', 'Bachelor', 'PhD']
Length: 4, dtype: str


tỷ lệ default của loan_grade


In [31]:
# Default rate theo loan_grade
grade_summary = df.groupby("loan_grade")["loan_status"].agg(["count", "mean"]).reset_index()
grade_summary.columns = ["loan_grade", "so_luong", "default_rate"]

grade_summary["default_rate_%"] = grade_summary["default_rate"] * 100

print(grade_summary.sort_values("loan_grade"))

  loan_grade  so_luong  default_rate  default_rate_%
0          A     10777        0.0996          9.9564
1          B     10451        0.1628         16.2760
2          C      6458        0.2073         20.7340
3          D      3626        0.5905         59.0458
4          E       964        0.6442         64.4191
5          F       241        0.7054         70.5394
6          G        64        0.9844         98.4375


ty lệ default của country

In [32]:
country_summary = df.groupby("country")["loan_status"].agg(["count", "mean"]).reset_index() 
country_summary.columns = ["country", "so_luong", "default_rate"]
country_summary["default_rate_%"] = country_summary["default_rate"] * 100
print(country_summary)

  country  so_luong  default_rate  default_rate_%
0  Canada     10785        0.2186         21.8637
1      UK     10944        0.2173         21.7288
2     USA     10852        0.2186         21.8577


tỷ lệ default theo: cb_person_default_on _file


In [34]:
default_history = df.groupby("cb_person_default_on_file")["loan_status"].agg(["count", "mean"]).reset_index()
default_history.columns = ["cb_person_default_on_file", "so_luong", "default_rate"]
default_history["default_rate_%"] = default_history["default_rate"] * 100
print(default_history)

  cb_person_default_on_file  so_luong  default_rate  default_rate_%
0                         N     26836        0.1839         18.3932
1                         Y      5745        0.3781         37.8068


 kiểm tra duplicate

In [24]:
df.duplicated().sum()

np.int64(0)

In [8]:
df["client_ID"].duplicated().sum()

np.int64(0)

kiểm tra outlier

In [5]:

# 1.person_age
print("1.person_age")
print("tuổi nhỏ nhất:", df["person_age"].min())
print("tuổi lớn nhất:", df["person_age"].max())
print("số tuổi quá nhỏ:", (df["person_age"] < 16).sum())

age_outlier = df[df["person_age"] > 80]
print("tổng số người quá  80 tuổi:", len(age_outlier))
print(age_outlier[["client_ID","person_age"]].head(10))

# 2.person_emp_length
print("2.person_emp_length")
print("sô năm làm nhỏ nhất:", df["person_emp_length"].min())
print("sô năm làm lớn nhất:", df["person_emp_length"].max())

emp_outlier = df[df["person_emp_length"] > 60]
print("tổng số người vượt quá  60 năm :", len(emp_outlier))
print(emp_outlier[["client_ID","person_emp_length"]].head(10))

# 3.tuổi và số năm làm việc
print("3.số năm làm việc lớn hơn tuổi")
logic_error = df[df["person_emp_length"] > (df["person_age"] - 14 )]
print("Số người làm việc trước 14 tuổi :", len(logic_error))
print(logic_error[["client_ID","person_age","person_emp_length"]].head(10))

# 4.person_income
print("4.thu nhập cá nhân")
print("min:", df["person_income"].min())
print("max:", df["person_income"].max())

income_outlier = df[df["person_income"] > 1000000]
print("Số người có thu nhập lớn hơn 1,000,000:", len(income_outlier))


#5.loan_to_income_ratio
print("5.tỷ lệ khoản vay trên thu nhập LTI")
print("min:", df["loan_to_income_ratio"].min())
print("max:", df["loan_to_income_ratio"].max())
print("Số dòng âm:", (df["loan_to_income_ratio"] < 0).sum())

lti_outlier = df[df["loan_to_income_ratio"] >= 1]
print("tổng số có LTI lớn hơn 1:", len(lti_outlier))

# 6.debt_to_income_ratio
print("6.tỷ lệ nợ trên thu nhập DTI")
print("min:", df["debt_to_income_ratio"].min())
print("max:", df["debt_to_income_ratio"].max())
print("Số dòng âm:", (df["debt_to_income_ratio"] < 0).sum())

dti_outlier = df[df["debt_to_income_ratio"] >= 1]
print("tổng số có DTI lớn hơn 1:", len(dti_outlier))
print(dti_outlier[["client_ID","debt_to_income_ratio"]].head(10))

# 7.loan_int_rate
print("7.tỷ lệ lãi suất")
print("min:", df["loan_int_rate"].min())
print("max:", df["loan_int_rate"].max())

rate_outlier = df[df["loan_int_rate"] > 20]
print("tổng số có lãi suất lớn hơn 20%:", len(rate_outlier))



1.person_age
tuổi nhỏ nhất: 20
tuổi lớn nhất: 144
số tuổi quá nhỏ: 0
tổng số người quá  80 tuổi: 7
        client_ID  person_age
81     CUST_00082         144
183    CUST_00184         144
575    CUST_00576         123
747    CUST_00748         123
32297  CUST_32298         144
32416  CUST_32417          94
32506  CUST_32507          84
2.person_emp_length
sô năm làm nhỏ nhất: 0.0
sô năm làm lớn nhất: 123.0
tổng số người vượt quá  60 năm : 2
      client_ID  person_emp_length
0    CUST_00001              123.0
210  CUST_00211              123.0
3.số năm làm việc lớn hơn tuổi
Số người làm việc trước 14 tuổi : 2
      client_ID  person_age  person_emp_length
0    CUST_00001          22              123.0
210  CUST_00211          21              123.0
4.thu nhập cá nhân
min: 4000
max: 6000000
Số người có thu nhập lớn hơn 1,000,000: 9
5.tỷ lệ khoản vay trên thu nhập LTI
min: 0.0007894736842
max: 0.83
Số dòng âm: 0
tổng số có LTI lớn hơn 1: 0
6.tỷ lệ nợ trên thu nhập DTI
min: 0.06450226791


phân loại kiểu dữ liệu và vai trò

In [4]:
VAR_META = {
    # (group, var_type)
    'client_ID'                  : ('ID',          'string'       ),
    'person_age'                 : ('Personal',     'numeric'    ),
    'person_income'              : ('Economic',     'numeric'     ),
    'person_home_ownership'      : ('Economic',     'categorical'),
    'person_emp_length'          : ('Economic',     'numeric'    ),
    'loan_intent'                : ('Loan',         'categorical'),
    'loan_grade'                 : ('Loan',         'categorical'   ),
    'loan_amnt'                  : ('Loan',         'numeric'   ),
    'loan_int_rate'              : ('Loan',         'numeric'    ),
    'loan_status'                : ('Target',       'binary'     ),
    'loan_percent_income'        : ('Loan',         'numeric'    ),
    'cb_person_default_on_file'  : ('Credit',       'categorical' ),
    'cb_person_cred_hist_length' : ('Credit',       'numeric'   ),
    'gender'                     : ('Personal',     'categorical'),
    'marital_status'             : ('Personal',     'categorical' ),
    'education_level'            : ('Personal',     'categorical' ),
    'country'                    : ('Geographic',   'categorical'),
    'state'                      : ('Geographic',   'categorical' ),
    'city'                       : ('Geographic',   'categorical'),
    'city_latitude'              : ('Geographic',   'numeric'   ),
    'city_longitude'             : ('Geographic',   'numeric'     ),
    'employment_type'            : ('Economic',     'categorical'),
    'loan_term_months'           : ('Loan',         'categorical' ),
    'loan_to_income_ratio'       : ('Risk',         'numeric'    ),
    'other_debt'                 : ('Risk',         'numeric'    ),
    'debt_to_income_ratio'       : ('Risk',         'numeric'     ),
    'open_accounts'              : ('Credit',       'numeric'    ),
    'credit_utilization_ratio'   : ('Risk',         'numeric'    ),
    'past_delinquencies'         : ('Credit',       'numeric'    ),
}
meta_df = pd.DataFrame.from_dict(
    VAR_META,
    orient='index',
    columns=['Group', 'VarType']
)

#Đưa tên biến từ index thành một cột bình thường
meta_df = meta_df.reset_index()

#Đổi tên cột 'index' thành 'Column'
meta_df = meta_df.rename(columns={'index': 'Column'})

#Xem  DataFrame
print("Bảng metadata:")
display(meta_df)

#Hiển thị theo từng nhóm
group_list = meta_df['Group'].unique()


Bảng metadata:


,Column,Group,VarType
0,client_ID,ID,string
1,person_age,Personal,numeric
2,person_income,Economic,numeric
3,person_home_ownership,Economic,categorical
4,person_emp_length,Economic,numeric
5,loan_intent,Loan,categorical
6,loan_grade,Loan,categorical
7,loan_amnt,Loan,numeric
8,loan_int_rate,Loan,numeric
9,loan_status,Target,binary
